# Train and package a UPAR 2027 Track 2 ConvNeXt Tiny submission

Run this notebook from top to bottom in Kaggle with **Internet** and a **GPU** enabled. It first downloads the challenge training and validation images and annotations, then fine-tunes an ImageNet-pretrained ConvNeXt Tiny on the 40 provided attributes. It saves `/kaggle/working/upar_track2_convnext_tiny_best.pth` and packages `/kaggle/working/upar_track2_convnext_tiny_submission.zip` for Codabench. Download the ZIP from Kaggle Outputs and upload it to the Track 2 submission page.

The download requires enough free disk space for the image archives and extracted files. Training 97,669 images can take substantial time. Re-running the download cells skips images already present. Only the provided train and validation splits are used. The Codabench runner has no Internet access; the ZIP contains the trained weights and uses only the container's built-in PyTorch, TorchVision, NumPy, and Pillow packages.


## 1. Download challenge data

These cells are copied from `UPAR-Challenge-2027/kaggle_download_datasets.ipynb`. They download Market1501, PA-100K, and PETA and copy the challenge annotations into `/kaggle/working/upar_2027/data`. Source dataset licenses and usage restrictions apply.


In [1]:
%pip -q install gdown tqdm requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
import csv
import shutil
import subprocess
import zipfile
from pathlib import Path

import gdown
import requests
from tqdm.auto import tqdm

WORK = Path('/kaggle/working')
REPO = WORK / 'UPAR-Challenge-2027'
DATA = WORK / 'upar_2027' / 'data'
DATA.mkdir(parents=True, exist_ok=True)

if not (REPO / 'peta_file_mapping.txt').exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/speckean/UPAR-Challenge-2027.git', str(REPO)
    ], check=True)
for split in ('train', 'val'):
    destination = DATA / 'annotations' / f'{split}.csv'
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(REPO / 'data' / 'annotations' / f'{split}.csv', destination)
print('Data directory:', DATA)

Cloning into '/kaggle/working/UPAR-Challenge-2027'...


Data directory: /kaggle/working/upar_2027/data


In [3]:
# The annotation paths determine exactly which images must be available.
needed = {'Market1501': set(), 'PA100k': set(), 'PETA': set()}
for split in ('train', 'val'):
    with (DATA / 'annotations' / f'{split}.csv').open(newline='') as handle:
        for row in list(csv.reader(handle))[1:]:
            relative = Path(row[0])
            needed[relative.parts[0]].add(relative)

def missing(source):
    return [path for path in needed[source] if not (DATA / path).is_file()]

def check_zip(path):
    if path.exists() and not zipfile.is_zipfile(path):
        path.unlink()  # Remove an interrupted download before retrying.
    return path.exists()

def require_complete(source):
    absent = missing(source)
    if absent:
        raise RuntimeError(f'{source}: {len(absent)} images missing; examples: {absent[:5]}')
    print(f'{source}: {len(needed[source]):,} annotated images available')

print({source: len(paths) for source, paths in needed.items()})

{'Market1501': 26458, 'PA100k': 88987, 'PETA': 15631}


## Market1501

In [4]:
market_archive = DATA / 'market_1501.zip'
market_dir = DATA / 'Market1501'
if missing('Market1501'):
    if not check_zip(market_archive):
        result = gdown.download(
            'https://drive.google.com/file/d/0B8-rUzbwVRk0c054eEozWG9COHM/view?resourcekey=0-8nyl7K9_x37HlQm34MmrYQ',
            output=str(market_archive), quiet=False, use_cookies=False, fuzzy=True
        )
        if not result or not zipfile.is_zipfile(market_archive):
            raise RuntimeError('Market1501 download failed or is not a ZIP file')
    with zipfile.ZipFile(market_archive) as archive:
        archive.extractall(DATA)
    extracted = DATA / 'Market-1501-v15.09.15'
    if extracted.exists():
        shutil.copytree(extracted, market_dir, dirs_exist_ok=True)
        shutil.rmtree(extracted)
require_complete('Market1501')
market_archive.unlink(missing_ok=True)

Downloading...
From (original): https://drive.google.com/uc?id=0B8-rUzbwVRk0c054eEozWG9COHM
From (redirected): https://drive.google.com/uc?id=0B8-rUzbwVRk0c054eEozWG9COHM&confirm=t&uuid=1c4dd0ec-14a2-49f4-b577-52e83180367d
To: /kaggle/working/upar_2027/data/market_1501.zip
100%|██████████| 153M/153M [00:03<00:00, 39.8MB/s]


Market1501: 26,458 annotated images available


## PA-100K

In [5]:
pa_dir = DATA / 'PA100k'
pa_dir.mkdir(exist_ok=True)
pa_archive = pa_dir / 'data.zip'
if missing('PA100k'):
    if not check_zip(pa_archive):
        gdown.download_folder(
            'https://drive.google.com/drive/folders/1d_D0Yh7C262gr0ef9EqkvG_M3fqgAWa2?usp=sharing',
            output=str(pa_dir), quiet=False, use_cookies=False
        )
        if not zipfile.is_zipfile(pa_archive):
            raise RuntimeError('PA-100K data.zip was not downloaded')
    with zipfile.ZipFile(pa_archive) as archive:
        archive.extractall(pa_dir)
require_complete('PA100k')
pa_archive.unlink(missing_ok=True)

Retrieving folder contents


Processing file 1dt5nteusNfCVrjAkffLK1sbh4hTLM8tl annotation.zip
Processing file 1Bod3MrbdCRiSvpR7NSOYi-rF3-ushwtj data.zip
Processing file 1zmQkoN2YJgbclT990ihj_DYU29YAidey README.txt


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1dt5nteusNfCVrjAkffLK1sbh4hTLM8tl
To: /kaggle/working/upar_2027/data/PA100k/annotation.zip
100%|██████████| 339k/339k [00:00<00:00, 19.4MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1Bod3MrbdCRiSvpR7NSOYi-rF3-ushwtj
From (redirected): https://drive.google.com/uc?id=1Bod3MrbdCRiSvpR7NSOYi-rF3-ushwtj&confirm=t&uuid=38321d57-5f88-4349-bfc4-ec58f7055dd8
To: /kaggle/working/upar_2027/data/PA100k/data.zip
100%|██████████| 451M/451M [00:04<00:00, 93.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1zmQkoN2YJgbclT990ihj_DYU29YAidey
To: /kaggle/working/upar_2027/data/PA100k/README.txt
100%|██████████| 794/794 [00:00<00:00, 4.50MB/s]
Download completed


PA100k: 88,987 annotated images available


## PETA

Only the mapped images are extracted from the archive, directly to their challenge filenames.

In [6]:
peta_dir = DATA / 'PETA'
peta_dir.mkdir(exist_ok=True)
peta_archive = peta_dir / 'peta.zip'
if missing('PETA'):
    if not check_zip(peta_archive):
        url = 'https://www.dropbox.com/s/52ylx522hwbdxz6/PETA.zip?dl=1'
        temporary = peta_archive.with_suffix('.zip.part')
        with requests.get(url, stream=True, timeout=(30, 120)) as response:
            response.raise_for_status()
            total = int(response.headers.get('content-length', 0))
            with temporary.open('wb') as output, tqdm(total=total or None, unit='B', unit_scale=True, desc='PETA') as bar:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        output.write(chunk)
                        bar.update(len(chunk))
        temporary.replace(peta_archive)
        if not zipfile.is_zipfile(peta_archive):
            raise RuntimeError('PETA download is not a ZIP file')
    with (REPO / 'peta_file_mapping.txt').open(newline='') as handle:
        mapping = list(csv.reader(handle))
    with zipfile.ZipFile(peta_archive) as archive:
        members = set(archive.namelist())
        for source, destination in tqdm(mapping, desc='Extracting PETA images'):
            target = DATA / destination
            if target.is_file():
                continue
            member = source.removeprefix('PETA/')
            if member not in members:
                raise FileNotFoundError(f'PETA archive is missing {member}')
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as original, target.open('wb') as output:
                shutil.copyfileobj(original, output)
require_complete('PETA')
peta_archive.unlink(missing_ok=True)

PETA:   0%|          | 0.00/236M [00:00<?, ?B/s]

Extracting PETA images:   0%|          | 0/19000 [00:00<?, ?it/s]

PETA: 15,631 annotated images available


In [7]:
for source in needed:
    require_complete(source)
print('Ready:', DATA)
print('Train annotations:', DATA / 'annotations' / 'train.csv')
print('Validation annotations:', DATA / 'annotations' / 'val.csv')

Market1501: 26,458 annotated images available
PA100k: 88,987 annotated images available
PETA: 15,631 annotated images available
Ready: /kaggle/working/upar_2027/data
Train annotations: /kaggle/working/upar_2027/data/annotations/train.csv
Validation annotations: /kaggle/working/upar_2027/data/annotations/val.csv


## 2. Training configuration

The download cells above create `DATA_ROOT` at `/kaggle/working/upar_2027/data`. Edit training settings here as needed. Pretrained weights are downloaded during Kaggle training only. Keep Internet enabled for the first run, or use a cached TorchVision checkpoint. The submitted `run.py` always constructs the model with `weights=None` before loading the checkpoint inside the ZIP.


In [8]:
"""Kaggle training script for UPAR 2027 Track 2.

Reads challenge data from /kaggle/working/upar_2027/data by default.
The submitted inference code and checkpoint are generated by this notebook.
"""

import csv
import importlib.util
import os
import random
import shutil
import tempfile
import zipfile
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms


DATA_ROOT = Path(os.environ.get("UPAR_DATA_ROOT", "/kaggle/working/upar_2027/data"))
OUTPUT_ROOT = Path("/kaggle/working" if Path("/kaggle/working").exists() else ".")
EPOCHS = 8
BATCH_SIZE = 64
NUM_WORKERS = 2
LEARNING_RATE = 1e-4
USE_IMAGENET_WEIGHTS = True  # Kaggle training only; no runtime download.
IMAGE_SIZE = (256, 128)  # height, width
SEED = 42




## 3. Read annotations and images

The released training CSV header omits the image-column label; this parser handles both train and validation formats.


In [9]:
def read_annotations(path):
    with path.open(newline="") as handle:
        reader = csv.reader(handle)
        header = next(reader)
        # The released train.csv omits the image-column name in its header.
        names = header if len(header) == 40 else header[1:]
        names = [name.removeprefix("# ").strip() for name in names]
        if len(names) != 40:
            raise ValueError(f"Expected 40 attributes in {path}, got {len(names)}")
        paths, labels = [], []
        for row_number, row in enumerate(reader, 2):
            if len(row) != 41:
                raise ValueError(f"{path}:{row_number}: expected image path and 40 labels")
            paths.append(row[0])
            labels.append([int(value) for value in row[1:]])
    labels = np.asarray(labels, dtype=np.float32)
    if not np.isin(labels, [0, 1]).all():
        raise ValueError(f"{path} contains labels other than 0 and 1")
    return names, paths, labels


class AttributeDataset(Dataset):
    def __init__(self, root, paths, labels, transform):
        self.root, self.paths, self.labels, self.transform = root, paths, labels, transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.root / self.paths[index]) as image:
            tensor = self.transform(image.convert("RGB"))
        return tensor, torch.from_numpy(self.labels[index])




## 4. Train and select the best checkpoint

The model predicts 40 independent attribute probabilities. Validation uses the provided validation split; the hidden test set is never accessed.


In [10]:
def main():
    if not (DATA_ROOT / "annotations/train.csv").is_file():
        raise FileNotFoundError(
            f"Missing {DATA_ROOT / 'annotations/train.csv'}. Run the download cells above "
            "in this Kaggle session or attach its saved output and set UPAR_DATA_ROOT."
        )
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.backends.cudnn.benchmark = True

    names, train_paths, train_labels = read_annotations(DATA_ROOT / "annotations/train.csv")
    val_names, val_paths, val_labels = read_annotations(DATA_ROOT / "annotations/val.csv")
    if names != val_names:
        raise ValueError("Train and validation attribute orders differ")
    for relative in train_paths[:10] + val_paths[:10]:
        if not (DATA_ROOT / relative).is_file():
            raise FileNotFoundError(f"Missing image {DATA_ROOT / relative}; finish downloading the images")
    print(f"Loaded {len(train_paths):,} train and {len(val_paths):,} validation images")

    normalize = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    train_transform = transforms.Compose([
        transforms.Resize(IMAGE_SIZE),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ])
    val_transform = transforms.Compose([
        transforms.Resize(IMAGE_SIZE), transforms.ToTensor(), normalize,
    ])
    train_loader = DataLoader(
        AttributeDataset(DATA_ROOT, train_paths, train_labels, train_transform),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        AttributeDataset(DATA_ROOT, val_paths, val_labels, val_transform),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    weights = models.ConvNeXt_Tiny_Weights.DEFAULT if USE_IMAGENET_WEIGHTS else None
    model = models.convnext_tiny(weights=weights)
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 40)
    model.to(device)
    positives = train_labels.sum(axis=0)
    negative_to_positive = (len(train_labels) - positives) / np.maximum(positives, 1)
    pos_weight = torch.tensor(np.clip(negative_to_positive, 1, 20), dtype=torch.float32, device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")

    checkpoint_path = OUTPUT_ROOT / "upar_track2_convnext_tiny_best.pth"
    best_loss = float("inf")
    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_total = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
                loss = criterion(model(images), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_total += loss.item() * len(images)

        model.eval()
        val_total = 0.0
        with torch.inference_mode():
            for images, labels in val_loader:
                images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
                    loss = criterion(model(images), labels)
                val_total += loss.item() * len(images)
        scheduler.step()
        train_loss = train_total / len(train_paths)
        val_loss = val_total / len(val_paths)
        print(f"Epoch {epoch}/{EPOCHS}: train BCE={train_loss:.4f}, val BCE={val_loss:.4f}")
        if val_loss < best_loss:
            best_loss = val_loss
            torch.save({
                "state_dict": {key: value.detach().cpu() for key, value in model.state_dict().items()},
                "attribute_names": names,
                "image_size": IMAGE_SIZE,
                "epoch": epoch,
                "val_loss": val_loss,
            }, checkpoint_path)
            print(f"Saved {checkpoint_path}")

    package_submission(checkpoint_path, DATA_ROOT / val_paths[0], names, val_labels[0])




## 5. Build the inference package

The generated `run.py` loads the checkpoint, predicts gallery attributes in batches, and returns a query-by-gallery distance matrix.


In [11]:
RUN_PY = '''"""UPAR 2027 Track 2 inference entry point."""
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch import nn
from torchvision import models, transforms

HERE = Path(__file__).resolve().parent
MODEL = None
ATTRIBUTE_NAMES = None
TRANSFORM = None
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_model():
    global MODEL, ATTRIBUTE_NAMES, TRANSFORM
    checkpoint = torch.load(HERE / "assets/upar_track2_convnext_tiny_best.pth", map_location="cpu", weights_only=True)
    ATTRIBUTE_NAMES = checkpoint["attribute_names"]
    size = tuple(checkpoint["image_size"])
    TRANSFORM = transforms.Compose([
        transforms.Resize(size),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    model = models.convnext_tiny(weights=None)
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 40)
    model.load_state_dict(checkpoint["state_dict"])
    MODEL = model.to(DEVICE).eval()


def rank_gallery(sample):
    if MODEL is None:
        load_model()
    order = [ATTRIBUTE_NAMES.index(name) for name in sample["attribute_names"]]
    gallery = sample["gallery"]
    queries = np.asarray(sample["queries"], dtype=np.float32)
    scores = np.empty((len(queries), len(gallery)), dtype=np.float32)
    batch_size = 64
    with torch.inference_mode():
        for start in range(0, len(gallery), batch_size):
            batch = gallery[start:start + batch_size]
            images = []
            for entry in batch:
                with Image.open(entry["image_path"]) as image:
                    images.append(TRANSFORM(image.convert("RGB")))
            logits = MODEL(torch.stack(images).to(DEVICE))
            probabilities = logits.sigmoid().float().cpu().numpy()[:, order]
            # L1 distance from each binary query to each predicted attribute vector.
            distances = probabilities.sum(axis=1)[None, :] + queries @ (1.0 - 2.0 * probabilities).T
            scores[:, start:start + len(batch)] = distances
    return {"distances": scores}
'''

METADATA_YAML = '''# Code submissions are executed by the challenges ingestion program, which
# imports run.py and calls rank_gallery(). Nothing in this
# file is used for inference, it exists only so the platform never treats the
# archive as a result submission. Leave it as-is.
command: python3 -c "print('Submission package ready. The ingestion program imports run.py')"
'''


def package_submission(checkpoint_path, sample_image, names, sample_query):
    folder = OUTPUT_ROOT / "upar_track2_convnext_tiny_submission"
    assets = folder / "assets"
    assets.mkdir(parents=True, exist_ok=True)
    (folder / "run.py").write_text(RUN_PY)
    (folder / "metadata.yaml").write_text(METADATA_YAML)
    shutil.copy2(checkpoint_path, assets / checkpoint_path.name)
    archive_path = OUTPUT_ROOT / "upar_track2_convnext_tiny_submission.zip"
    expected = {"run.py", "metadata.yaml", "assets/upar_track2_convnext_tiny_best.pth"}
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for name in sorted(expected):
            archive.write(folder / name, name)
    with zipfile.ZipFile(archive_path) as archive:
        print("ZIP contents:", archive.namelist())
        assert set(archive.namelist()) == expected
        with tempfile.TemporaryDirectory() as temp_dir:
            archive.extractall(temp_dir)
            spec = importlib.util.spec_from_file_location("upar_submission_check", Path(temp_dir) / "run.py")
            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)
            result = module.rank_gallery({
                "attribute_names": names,
                "queries": [sample_query.astype(int).tolist()],
                "gallery": [{"index": 0, "image_path": str(sample_image)}],
            })
            distances = np.asarray(result["distances"])
            assert distances.shape == (1, 1) and np.isfinite(distances).all()
            print("Packaged inference smoke test passed")
    print("Checkpoint:", checkpoint_path)
    print("Upload this ZIP to Codabench:", archive_path)




## 6. Run training and create the ZIP

After this cell finishes, find the `.pth` checkpoint and submission ZIP under `/kaggle/working`. The lowest validation weighted BCE selects the checkpoint. After upload, compare its mADM with the ResNet-18 baseline (0.3469); a stronger backbone does not guarantee a higher retrieval score.


In [12]:
main()


Loaded 97,669 train and 33,407 validation images
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 243MB/s] 
/tmp/ipykernel_24/935877455.py:53: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")
/tmp/ipykernel_24/935877455.py:63: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
/tmp/ipykernel_24/935877455.py:75: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=device.type == "cuda"):


Epoch 1/8: train BCE=0.3554, val BCE=0.3798
Saved /kaggle/working/upar_track2_convnext_tiny_best.pth
Epoch 2/8: train BCE=0.2185, val BCE=0.3756
Saved /kaggle/working/upar_track2_convnext_tiny_best.pth
Epoch 3/8: train BCE=0.1663, val BCE=0.3929
Epoch 4/8: train BCE=0.1302, val BCE=0.4175
Epoch 5/8: train BCE=0.1034, val BCE=0.4700
Epoch 6/8: train BCE=0.0839, val BCE=0.5336
Epoch 7/8: train BCE=0.0702, val BCE=0.5617
Epoch 8/8: train BCE=0.0632, val BCE=0.5758
ZIP contents: ['assets/upar_track2_convnext_tiny_best.pth', 'metadata.yaml', 'run.py']
Packaged inference smoke test passed
Checkpoint: /kaggle/working/upar_track2_convnext_tiny_best.pth
Upload this ZIP to Codabench: /kaggle/working/upar_track2_convnext_tiny_submission.zip
